#①ライブラリのインポート

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from torch.utils import data
import torchvision
from torchvision import datasets
from torchvision import transforms as T
from torchsummary import summary
import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
import time
import os
import math
print('pytorch',torch.__version__)
print('torchvision', torchvision.__version__)

#②シード固定の関数実装

In [ ]:
def init_seed(seed=0):
  np.random.seed(seed) #npライブラリシード値の固定
  torch.manual_seed(seed) #Pytorchのシード値の固定
  torch.cuda.manual_seed(seed) #PytorchのGPU上のシード値の固定
  torch.backends.cudnn.deterministic = True #cuDNNのアルゴリズムのランダム性を無くす
seed = 3407 #seed値を3407に設定

#③MNISTデータの取得

In [ ]:
image_size = 28
transform = T.Compose(
    [T.Resize((image_size,image_size)), #画像サイズを28x28にリサイズ
     T.ToTensor() #Pytorchのテンソルに変換し、値を0.0〜1.0にスケーリング
     ])
train_transform = T.Compose([
    T.Resize((image_size, image_size)),
    T.RandomRotation(5, fill=0),#回転を利用した学習用のデータ拡張
    T.ToTensor()
])
#MNISTから学習データ、テストデータを取得
train_dataset = datasets.MNIST(root='./data', train=True,
                               transform=train_transform, download = True)
test_dataset = datasets.MNIST(root='./data', train=False,
                              transform=transform, download = True)
train_samples = len(train_dataset)#学習データのサンプル数
classes = [key for key in test_dataset.class_to_idx]#テストデータセットから正解ラベルとクラス名を取り出す
print("train dataset:",train_dataset,"\n")
print("test dataset:\n",test_dataset,"\n")
print("dataset classes:\n",classes)

#④MNISTデータのバランスの可視化

In [ ]:
train_df = pd.DataFrame(train_dataset.targets)
balance = train_df.value_counts()#学習用データセットの正解ラベル数の集計
display('train',balance)
test_df = pd.DataFrame(test_dataset.targets)
balance = test_df.value_counts()#テスト用データセットの正解ラベル数の集計
display('test',balance)

#⑤データローダの作成

In [ ]:
batch_size = 128
init_seed(seed)
train_loader = data.DataLoader(train_dataset, batch_size = batch_size,
                               shuffle=True, num_workers = 2)
test_loader = data.DataLoader(test_dataset, batch_size=batch_size,
                              shuffle=False, num_workers = 2)

#⑥1.ロギング機能の実装

In [ ]:
class Logger():
  def __init__(self,print_iteration, train_samples, batch_size):
    self.print_iteration = print_iteration
    self.iterations = math.ceil(train_samples / batch_size)
    self.log = {'loss': [],'acc': [], 'epochs':[]}
  def clear_run(self):
    self.run_loss = 0.0
    self.run_acc = 0.0
    self.run_count = 0
    self.s_time = time.time()
  def logging(self, outputs, loss, labels, epoch, i):
    b_size = outputs.size(0)
    self.run_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    correct = (predicted == labels).sum()
    accuracy = 100 * correct / b_size
    self.run_acc += accuracy.item()
    self.run_count += 1
    if i % self.print_iteration == 0 or i == self.iterations:
      e_time = (time.time() - self.s_time) / self.run_count
      progress = round(epoch + i / self.iterations, 2)
      self.log['loss'].append(self.run_loss / self.run_count)
      self.log['acc'].append(self.run_acc / self.run_count)
      self.log['epochs'].append(progress)
      print('[%5d, %9d] loss:%.3f, accuracy:%.2f%%, speed:%.3fs/iter' % (epoch, i, self.log['loss'][-1], self.log['acc'][-1], e_time))
      self.clear_run()
  def save(self, path):
    dict_ = dict(epochs = self.log['epochs'], loss = self.log['loss'],accuracy = self.log['acc'])
    df = pd.DataFrame(dict_)
    df.to_csv(path)

#⑥2.精度検証機能の実装

In [ ]:
class Evaluator():
  def __init__(self, dataloader, criterion, device):
    self.log = {'loss': [],'acc': [], 'epochs':[]}
    self.loader = dataloader
    self.device = device
    self.n_iters = len(self.loader)
    self.criterion = criterion
  def evaluation(self, model, epoch):
    model.eval()
    correct = 0
    val_loss = 0.0
    val_accuracy = 0.0
    test_samples = 0
    eval_start = time.time()
    for images, labels in self.loader:
      images = images.to(self.device)
      labels = labels.to(self.device)
      b_size = images.size(0)
      test_samples += b_size
      with torch.no_grad():
        outputs = model(images)
      loss = self.criterion(outputs, labels)
      val_loss += loss.item()
      _, predicted = torch.max(outputs.data, 1)
      correct += (predicted == labels).sum().item()
    eval_time = (time.time() - eval_start) / self.n_iters
    val_accuracy = 100 * correct / test_samples
    self.log['loss'].append(val_loss / self.n_iters)
    self.log['acc'].append(val_accuracy)
    self.log['epochs'].append(epoch)
    print('[%5d]test loss: %.3f, accuracy: %.2f%%, speed:%.3fs/iter' % (epoch, self.log['loss'][-1], self.log['acc'][-1], eval_time))
  def save(self, path):
    dict_ = dict(epochs = self.log['epochs'], loss = self.log['loss'],accuracy = self.log['acc'])
    df = pd.DataFrame(dict_)
    df.to_csv(path)

#⑦CNNモデルの定義

In [ ]:
class MyCNN(nn.Module):
  def __init__(self):
    super(MyCNN, self).__init__()
    self.conv1 = nn.Conv2d(1, 50, (5, 5))
    self.conv2 = nn.Conv2d(50, 100, (3, 3), padding=1)
    self.fc1 = nn.Linear(6*6*100 , 100)
    #self.fc1 = nn.Linear(12*12*50 , 100)
    self.fc2 = nn.Linear(100, 10)
    self.pool = nn.MaxPool2d((2,2))
    self.flat = nn.Flatten()
    self.drop = nn.Dropout(0.5)
    self.relu = nn.ReLU()
  def forward(self, x):
    h = self.relu(self.conv1(x))#畳み込み層の後ReLU関数により活性化
    h = self.pool(h)#Maxプーリング層
    h = self.relu(self.conv2(h))#畳み込み層の後ReLU関数により活性化3.2 ⑰精度向上のために追加
    h = self.pool(h)#Maxプーリング層 3.2 ⑰精度向上のために追加
    h = self.flat(h)#平坦化
    h = self.relu(self.fc1(h))#全結合層の後ReLU関数により活性化
    h = self.drop(h)#ドロップアウト 3.2 ⑰精度向上のために追加
    y = self.fc2(h)#全結合層の後ReLU関数により活性化
    return y
  def predict(self, x):
    y = self.forward(x)
    return F.softmax(y, dim=1)#softmax関数を通して推論結果を出力

#⑧CNNのインスタンス生成、サマリの表示および損失関数、最適化手法の設定

In [ ]:
learning_rate = 0.001
init_seed(seed) #seedの値は任意に設定する
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('use device:', device)
model = MyCNN().to(device)
# image_size変数をtransform作成時に予め定義しておく
summary(model, (1, image_size, image_size),device=device)
criterion = nn.CrossEntropyLoss()#交差エントロピー誤差の計算
#optimizer = optim.SGD(model.parameters(), lr=learning_rate) #最適化手法SDG 3.2 ⑬の時はコメントアウトしておく
optimizer = optim.Adam(model.parameters(), lr=learning_rate) #最適化手法Adam 3.2 ⑬の時だけ使用する
print('optimizer',optimizer)

#⑨モデルの学習

📝最終エポックでのaccuracy、lossの値、学習に要した時間を記録する📝

In [ ]:
from sklearn.model_selection import StratifiedKFold
init_seed(1234) #ここでの seed の値は 1234 で固定する
skf = StratifiedKFold(n_splits=100)
split_set = skf.split(train_dataset.data, train_dataset.targets)
_, train_index = next(split_set)
sub_train_dataset = data.Subset(train_dataset, train_index)
sub_train_df = pd.DataFrame(train_dataset.targets[train_index])
sub_balance = sub_train_df.value_counts()
train_samples = len(sub_train_dataset)
display('sub train', sub_balance)

In [ ]:
train_transform = T.Compose(
    [T.Resize((image_size,image_size)), #画像サイズを28x28に
     T.RandomRotation(5,fill=0),#3.2 ⑰精度向上のために追加
     T.ToTensor() #Pytorchのテンソルに変換
     ])
train_dataset = datasets.MNIST(root='./data', train=True,
                               transform=transform, download = True)
batch_size = 128
#sub_train_loaderにsub_train_datasetを入れる
sub_train_loader = data.DataLoader(sub_train_dataset,
                                   batch_size = batch_size, shuffle=True, num_workers = 2)

In [ ]:
n_epoch=40#エポック数
print_iteration=5#学習時のロギング間隔
logger=Logger(print_iteration,train_samples,batch_size)
eval=Evaluator(test_loader,criterion,device)
start=time.time()
for epoch in tqdm(range(n_epoch)):#エポック数だけ学習を行う。
  eval.evaluation(model,epoch)
  model.train()
  logger.clear_run()
  print_time=time.time()
  for i,(images,labels) in enumerate(sub_train_loader,0):
    images=images.to(device)
    labels=labels.to(device)
    outputs=model(images)
    optimizer.zero_grad()
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    logger.logging(outputs,loss,labels,epoch,i+1)
eval.evaluation(model,n_epoch)
elapsed_time=time.time()-start
print('training total',elapsed_time,'sec.')#合計の処理時間

#データの記録
finalaccuracy = eval.log["acc"][-1] #accuracy
finalloss = eval.log["loss"][-1] # loss

#⑩モデルの保存



In [ ]:
base_fol_prefix = '/content/drive/MyDrive/C-4' #MyDrive内のモデルを保存したいところにパスを指定する
folder_name = f'Acc{finalaccuracy:.2f}_Loss{finalloss:.3f}_Time{elapsed_time:.0f}s' #accuracy、loss、時間の名前を付けてフォルダを作成
base_fol = os.path.join(base_fol_prefix, folder_name) #作成したフォルダ内にモデルを収納
os.makedirs(base_fol,exist_ok = True)
torch.save(model.state_dict(),os.path.join(base_fol,'mnist.pth'))
logger.save(os.path.join(base_fol,'train_loss_acc.csv'))
eval.save(os.path.join(base_fol,'test_loss_acc.csv'))
#テキストファイルにモデルと最適化手法を書き込んで保存
with open( os.path.join(base_fol,'architecture.txt'),mode = "w") as f:
  f.write('model\n')
  f.write(str(model))
  f.write('optimizer\n')
  f.write(str(optimizer))

#⑪学習結果の可視化

In [ ]:
mpl.rcParams['figure.dpi'] = 150
#モデルのエポックと精度のグラフのプロット
plt.plot(logger.log['epochs'], logger.log['acc'], marker="o",linestyle='--')
plt.plot(eval.log['epochs'], eval.log['acc'], marker="o")
plt.title('Model accuracy')
plt.ylabel('Accuracy[%]')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'])
plt.show()
plt.savefig(os.path.join(base_fol, 'accuracy_plot.png'))
#モデルのエポックと損失のグラフのプロット
plt.plot(logger.log['epochs'], logger.log['loss'], marker="o",linestyle='--')
plt.plot(eval.log['epochs'], eval.log['loss'], marker="o")
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'])
plt.show()
plt.savefig(os.path.join(base_fol, 'loss_plot.png'))#グラフの保存

#⑫1.ランタイムをGPUに変更

ランタイム➡ランタイムのタイプを変更➡T4 GPU

①から手順をやり直す

#⑫2.GPUの種類を調べる

In [ ]:
!nvidia-smi

#⑬最適化方法を変更

⑧のコメントアウトされている行を使う

⑨～⑪を再度行う

#⑭モデルのコンパイル、学習の別の例

In [ ]:
from sklearn.model_selection import StratifiedKFold
init_seed(1234) #ここでの seed の値は 1234 で固定する
skf = StratifiedKFold(n_splits=100)
split_set = skf.split(train_dataset.data, train_dataset.targets)
_, train_index = next(split_set)
sub_train_dataset = data.Subset(train_dataset, train_index)
sub_train_df = pd.DataFrame(train_dataset.targets[train_index])
sub_balance = sub_train_df.value_counts()
train_samples = len(sub_train_dataset)
display('sub train', sub_balance)

#⑮サブセット用のデータローダ作成の例

In [ ]:
transform = T.Compose(
    [T.Resize((image_size,image_size)), #画像サイズを28x28に
    # T.RandomRotation(5,fill=0),
     T.ToTensor() #Pytorchのテンソルに変換
     ])

batch_size = 128
#sub_train_loaderにsub_train_datasetを入れる
sub_train_loader = data.DataLoader(sub_train_dataset,
                                   batch_size = batch_size, shuffle=True, num_workers = 2)

#⑯サブデータセット用のデータローダを用いて再度⑦～⑫を行う

この時、⑧の最適化方法をSGDに戻しておくのを忘れない

#⑰精度を向上させる方法を検討する

必ず①何をしたか②精度は向上したかをメモすること